# Import

In [6]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence
from scipy import stats

# Load dataset

In [7]:
DATA_PATH = "Carseats.csv"

# -----------------------------
# Load & prep data
# -----------------------------
df = pd.read_csv(DATA_PATH)

# Ensure qualitative variables are treated as categorical
for col in ["Urban", "US"]:
    if col in df.columns:
        df[col] = df[col].astype("category")

# Quick sanity check
print("Rows, Cols:", df.shape)
print(df[["Sales", "Price", "Urban", "US"]].head())

Rows, Cols: (400, 11)
   Sales  Price Urban   US
0   9.50    120   Yes  Yes
1  11.22     83   Yes  Yes
2  10.06     80   Yes  Yes
3   7.40     97   Yes  Yes
4   4.15    128   Yes   No


# (a) Fit multiple regression: Sales ~ Price + Urban + US

In [8]:
# Use C() to explicitly treat Urban and US as categorical (dummy variables).
model_full = smf.ols("Sales ~ Price + C(Urban) + C(US)", data=df).fit()

print("\n(a) Full model summary:")
print(model_full.summary())


(a) Full model summary:
                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.239
Model:                            OLS   Adj. R-squared:                  0.234
Method:                 Least Squares   F-statistic:                     41.52
Date:                Sun, 25 Jan 2026   Prob (F-statistic):           2.39e-23
Time:                        20:06:42   Log-Likelihood:                -927.66
No. Observations:                 400   AIC:                             1863.
Df Residuals:                     396   BIC:                             1879.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept        

# (b) Interpret coefficients (print a simple interpretation)

In [10]:
# Baselines: the first category alphabetically by default (often 'No').
# You can see the coding in model_full.model.data.design_info.
params_full = model_full.params
print("\n(b) Coefficients (full model):")
print(params_full)

# Helper: compute baseline levels for reference
us_levels = list(df["US"].cat.categories) if "US" in df.columns else []
urban_levels = list(df["Urban"].cat.categories) if "Urban" in df.columns else []


(b) Coefficients (full model):
Intercept          13.043469
C(Urban)[T.Yes]    -0.021916
C(US)[T.Yes]        1.200573
Price              -0.054459
dtype: float64


# (c) Write out the model equation (handled conceptually)

In [11]:
print("\n(c) Model equation (conceptual):")
print("Sales = β0 + β1*Price + β2*Urban(Yes) + β3*US(Yes) + ε")


(c) Model equation (conceptual):
Sales = β0 + β1*Price + β2*Urban(Yes) + β3*US(Yes) + ε


# (d) Hypothesis testing: H0 : βj = 0

In [12]:
print("\n(d) Hypothesis tests for coefficients:")
print(model_full.pvalues)

significant_predictors = model_full.pvalues[model_full.pvalues < 0.05]
print("\nSignificant predictors at α = 0.05:")
print(significant_predictors)


(d) Hypothesis tests for coefficients:
Intercept          3.626602e-62
C(Urban)[T.Yes]    9.357389e-01
C(US)[T.Yes]       4.860245e-06
Price              1.609917e-22
dtype: float64

Significant predictors at α = 0.05:
Intercept       3.626602e-62
C(US)[T.Yes]    4.860245e-06
Price           1.609917e-22
dtype: float64


# (e) Fit reduced model with significant predictors only (Price and US)

In [13]:
model_reduced = smf.ols("Sales ~ Price + C(US)", data=df).fit()

print("\n(e) Reduced model summary:")
print(model_reduced.summary())


(e) Reduced model summary:
                            OLS Regression Results                            
Dep. Variable:                  Sales   R-squared:                       0.239
Model:                            OLS   Adj. R-squared:                  0.235
Method:                 Least Squares   F-statistic:                     62.43
Date:                Sun, 25 Jan 2026   Prob (F-statistic):           2.66e-24
Time:                        20:11:35   Log-Likelihood:                -927.66
No. Observations:                 400   AIC:                             1861.
Df Residuals:                     397   BIC:                             1873.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       13.0

# (f) Compare model fit: R^2, Adjusted R^2, AIC, BIC

In [14]:
comparison = pd.DataFrame({
    "Model": ["Full", "Reduced"],
    "R_squared": [model_full.rsquared, model_reduced.rsquared],
    "Adj_R_squared": [model_full.rsquared_adj, model_reduced.rsquared_adj],
    "AIC": [model_full.aic, model_reduced.aic],
    "BIC": [model_full.bic, model_reduced.bic]
})

print("\n(f) Model fit comparison:")
print(comparison)


(f) Model fit comparison:
     Model  R_squared  Adj_R_squared          AIC          BIC
0     Full   0.239275       0.233512  1863.312074  1879.277932
1  Reduced   0.239263       0.235430  1861.318648  1873.293042


# (g) 95% confidence intervals for reduced model

In [15]:
conf_int = model_reduced.conf_int(alpha=0.05)
conf_int.columns = ["Lower 95%", "Upper 95%"]

print("\n(g) 95% Confidence Intervals (Reduced Model):")
print(conf_int)


(g) 95% Confidence Intervals (Reduced Model):
              Lower 95%  Upper 95%
Intercept      11.79032  14.271265
C(US)[T.Yes]    0.69152   1.707766
Price          -0.06476  -0.044195


# (h) Outliers & high leverage diagnostics

In [16]:
influence = OLSInfluence(model_reduced)

# Studentized residuals
student_resid = influence.resid_studentized_external

# Leverage (hat values)
leverage = influence.hat_matrix_diag

# Cook's distance
cooks_d = influence.cooks_distance[0]

diagnostics = pd.DataFrame({
    "Studentized_Residual": student_resid,
    "Leverage": leverage,
    "Cooks_Distance": cooks_d
})

# Thresholds
n = df.shape[0]
p = model_reduced.df_model + 1  # predictors + intercept
leverage_threshold = 2 * p / n
cooks_threshold = 4 / n

print("\n(h) Potential outliers or influential points:")
print("Leverage threshold:", leverage_threshold)
print("Cook's Distance threshold:", cooks_threshold)

flagged = diagnostics[
    (np.abs(diagnostics["Studentized_Residual"]) > 3) |
    (diagnostics["Leverage"] > leverage_threshold) |
    (diagnostics["Cooks_Distance"] > cooks_threshold)
]

print("\nFlagged observations (if any):")
print(flagged.head())
print("\nTotal flagged points:", flagged.shape[0])


(h) Potential outliers or influential points:
Leverage threshold: 0.015
Cook's Distance threshold: 0.01

Flagged observations (if any):
    Studentized_Residual  Leverage  Cooks_Distance
25              2.599652  0.011622        0.026109
28             -2.433745  0.005636        0.011054
30              2.194942  0.009835        0.015799
42             -0.534993  0.043338        0.004330
49              2.334376  0.012553        0.022835

Total flagged points: 35
